In [41]:
# =============================================================================
# Sensor Fusion for Depth Estimation (KITTI) – Late Fusion Demonstration
# =============================================================================
#
# Author: Vasu Tammisetti
# Date: March 2026
#
# Project Goal:
#   Demonstrate sensor fusion by combining sparse LiDAR depth with dense
#   monocular depth (MiDaS) to produce an accurate, dense depth map.
#
# Key Techniques:
#   - LiDAR projection to image plane using KITTI calibration.
#   - Nearest‑neighbor interpolation to densify LiDAR depth.
#   - Monocular depth estimation with a pre‑trained transformer (MiDaS).
#   - Late fusion via median scaling and averaging.
#
# Dataset: KITTI Object Detection (training set, first 100 samples)
#   - Images:      data_object_image_2/training/
#   - LiDAR:       data_object_velodyne/training/
#   - Calibration: data_object_calib/training/
#
# Why This Matters:
#   Accurate dense depth is critical for autonomous driving, robotics,
#   and AR/VR. Fusion leverages the strengths of each sensor:
#     • LiDAR – precise but sparse.
#     • Camera – dense but scale‑ambiguous.
#
# =============================================================================

In [43]:
# Step 1: Install required packages
!pip install torch torchvision opencv-python matplotlib scipy tqdm imageio -q

In [44]:
# Step 2: Import all necessary libraries
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from scipy.spatial import KDTree
from tqdm import tqdm
import imageio
import glob
import zipfile
from google.colab import files
from google.colab import drive

In [ ]:
# Cell 3: Mount Google Drive if you want to save results there (optional)
#drive.mount('/content/drive')

In [45]:
# Step  4: Define paths to your KITTI training folders
# Update base_path if your data is located elsewhere
base_path = '/content/sensorfusion_data/sensorfusion/'

image_dir = os.path.join(base_path, 'data_object_image_2', 'training')
velo_dir  = os.path.join(base_path, 'data_object_velodyne', 'training')
calib_dir = os.path.join(base_path, 'data_object_calib', 'training')

# Verify folders exist
for d in [image_dir, velo_dir, calib_dir]:
    if not os.path.exists(d):
        raise FileNotFoundError(f"Folder not found: {d}\nPlease adjust base_path.")
print("✅ All training folders found.")

✅ All training folders found.


In [47]:
# Step 5: Get sorted file lists and keep only the first 100 (Due large volume of original data)
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
velo_files  = sorted([f for f in os.listdir(velo_dir)  if f.endswith('.bin')])
calib_files = sorted([f for f in os.listdir(calib_dir) if f.endswith('.txt')])

# Limit to first 100 (or fewer if less exist)
N = 100
image_files = image_files[:N]
velo_files  = velo_files[:N]
calib_files = calib_files[:N]

print(f"📁 Using {len(image_files)} images, {len(velo_files)} point clouds, {len(calib_files)} calibration files.")

📁 Using 100 images, 100 point clouds, 100 calibration files.


In [48]:
# step 6: Helper function to load calibration data for a given frame index
def load_calib_for_frame(calib_dir, idx):
    """
    Load P2, R0_rect, and Tr_velo_to_cam from the calibration file.
    """
    calib_path = os.path.join(calib_dir, f'{idx:06d}.txt')
    with open(calib_path, 'r') as f:
        lines = f.readlines()

    P2 = np.array([float(x) for x in lines[2].strip().split()[1:]]).reshape(3, 4)
    R0_rect = np.array([float(x) for x in lines[4].strip().split()[1:]]).reshape(3, 3)
    Tr_velo_to_cam = np.array([float(x) for x in lines[5].strip().split()[1:]]).reshape(3, 4)

    return P2, R0_rect, Tr_velo_to_cam

In [49]:
# step 7: Helper function to project LiDAR points onto the image plane
def project_velo_to_image(velo_points, P2, R0_rect, Tr_velo_to_cam):
    """
    velo_points: N x 3 array (x, y, z) in velodyne coordinates.
    Returns: (pts_img, depths) where pts_img are (u,v) pixel coordinates,
             depths are depths in camera coordinates.
    """
    # Transform to camera coordinates
    pts_cam = (R0_rect @ (Tr_velo_to_cam[:, :3] @ velo_points.T + Tr_velo_to_cam[:, 3:4])).T

    # Project to image plane
    pts_img_hom = (P2 @ np.vstack((pts_cam.T, np.ones((1, pts_cam.shape[0]))))).T

    # Normalize
    pts_img = np.zeros_like(pts_img_hom[:, :2])
    pts_img[:, 0] = pts_img_hom[:, 0] / pts_img_hom[:, 2]   # u
    pts_img[:, 1] = pts_img_hom[:, 1] / pts_img_hom[:, 2]   # v

    return pts_img, pts_cam[:, 2]

In [51]:
# step 8: Nearest‑neighbor interpolation of sparse depth map
def interpolate_lidar_depth(sparse_depth, image_shape):
    h, w = image_shape
    valid_pixels = np.column_stack(np.where(sparse_depth > 0))
    valid_depths = sparse_depth[sparse_depth > 0]

    if len(valid_pixels) == 0:
        return np.zeros((h, w))

    tree = KDTree(valid_pixels)
    grid_y, grid_x = np.mgrid[0:h, 0:w]
    all_pixels = np.column_stack((grid_y.ravel(), grid_x.ravel()))
    distances, indices = tree.query(all_pixels)

    dense_depth = valid_depths[indices].reshape(h, w)
    return dense_depth

In [52]:
# Step 9: Load pre-trained MiDaS model for monocular depth estimation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

model_type = "DPT_Large"   # Best quality
midas = torch.hub.load("intel-isl/MiDaS", model_type)
midas.to(device)
midas.eval()

# Load transforms
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
if model_type in ["DPT_Large", "DPT_Hybrid"]:
    transform = midas_transforms.dpt_transform
else:
    transform = midas_transforms.small_transform

🚀 Using device: cuda


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [53]:
# step 10: Helper to compute monocular depth for an image
def get_monocular_depth(image_bgr):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    return prediction.cpu().numpy()

In [54]:
# step 11: Create folders to save results
results_dir = '/content/fusion_results'
gif_frames_dir = '/content/gif_frames'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(gif_frames_dir, exist_ok=True)

In [65]:
# Cell 12: Process frames, generate six‑panel images with clear benchmarking
N = 10   # Number of frames to process (change to 100 for full dataset)

print(f"\n🔄 Processing {N} frames and saving results...\n")

for idx in range(N):
    print(f"  Frame {idx:06d}")

    # File paths
    img_path = os.path.join(image_dir, f'{idx:06d}.png')
    velo_path = os.path.join(velo_dir, f'{idx:06d}.bin')
    calib_path = os.path.join(calib_dir, f'{idx:06d}.txt')

    if not (os.path.exists(img_path) and os.path.exists(velo_path) and os.path.exists(calib_path)):
        print(f"   ⚠️ Missing files for index {idx}, skipping.")
        continue

    # Load calibration
    P2, R0_rect, Tr_velo_to_cam = load_calib_for_frame(calib_dir, idx)

    # Load image
    image_bgr = cv2.imread(img_path)
    if image_bgr is None:
        print(f"   ⚠️ Could not read image {img_path}")
        continue
    h, w, _ = image_bgr.shape

    # Load LiDAR
    velo = np.fromfile(velo_path, dtype=np.float32).reshape(-1, 4)
    points = velo[:, :3]

    # Project LiDAR
    pts, depths = project_velo_to_image(points, P2, R0_rect, Tr_velo_to_cam)
    mask = (pts[:, 0] >= 0) & (pts[:, 0] < w) & (pts[:, 1] >= 0) & (pts[:, 1] < h) & (depths > 0)
    pts = pts[mask]
    depths = depths[mask]

    # Sparse depth
    sparse_depth = np.zeros((h, w), dtype=np.float32)
    for (u, v), d in zip(pts.astype(int), depths):
        sparse_depth[v, u] = d

    # Dense LiDAR depth
    lidar_dense = interpolate_lidar_depth(sparse_depth, (h, w))

    # Monocular depth
    mono_depth = get_monocular_depth(image_bgr)

    # Scale monocular depth to match LiDAR median
    valid_lidar = lidar_dense[lidar_dense > 0]
    if len(valid_lidar) > 0:
        lidar_median = np.median(valid_lidar)
        mono_median = np.median(mono_depth)
        mono_scaled = mono_depth * (lidar_median / mono_median) if mono_median > 0 else mono_depth
    else:
        mono_scaled = mono_depth

    # Fused depth (average)
    fused_depth = (mono_scaled + lidar_dense) / 2

    # ---- SIX-PANEL FIGURE WITH ENHANCED BENCHMARKING ----
    fig = plt.figure(figsize=(18, 12))

    # 1. Camera image
    plt.subplot(2, 3, 1)
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title('(a) Camera Image (Reference)', fontsize=12, fontweight='bold')
    plt.axis('off')

    # 2. Sparse LiDAR depth (metric, 0-50m)
    plt.subplot(2, 3, 2)
    im2 = plt.imshow(sparse_depth, cmap='plasma', vmin=0, vmax=50)
    plt.title('(b) Sparse LiDAR Depth\n(Accurate but sparse)', fontsize=12, fontweight='bold')
    plt.colorbar(im2, fraction=0.046, pad=0.04, label='Depth (m)')
    plt.axis('off')

    # 3. Dense LiDAR depth (metric, 0-50m)
    plt.subplot(2, 3, 3)
    im3 = plt.imshow(lidar_dense, cmap='plasma', vmin=0, vmax=50)
    plt.title('(c) Dense LiDAR Depth (interpolated)\n(Fills gaps, may blur edges)', fontsize=12, fontweight='bold')
    plt.colorbar(im3, fraction=0.046, pad=0.04, label='Depth (m)')
    plt.axis('off')

    # 4. Raw monocular depth (arbitrary scale)
    plt.subplot(2, 3, 4)
    im4 = plt.imshow(mono_depth, cmap='plasma')
    plt.title('(d) Monocular Depth (MiDaS raw)\n(Dense but scale‑ambiguous)', fontsize=12, fontweight='bold')
    plt.colorbar(im4, fraction=0.046, pad=0.04, label='Relative depth')
    plt.axis('off')

    # 5. Scaled monocular depth (metric, 0-50m)
    plt.subplot(2, 3, 5)
    im5 = plt.imshow(mono_scaled, cmap='plasma', vmin=0, vmax=50)
    plt.title('(e) Monocular Depth (scaled)\n(Now metric, preserves edges)', fontsize=12, fontweight='bold')
    plt.colorbar(im5, fraction=0.046, pad=0.04, label='Depth (m)')
    plt.axis('off')

    # 6. Fused depth (metric, 0-50m)
    plt.subplot(2, 3, 6)
    im6 = plt.imshow(fused_depth, cmap='plasma', vmin=0, vmax=50)
    plt.title('(f) Fused Depth (average)\n✅ Best of both: accurate + dense', fontsize=12, fontweight='bold')
    plt.colorbar(im6, fraction=0.046, pad=0.04, label='Depth (m)')
    plt.axis('off')

    plt.suptitle(f'KITTI Frame {idx:06d} – Sensor Fusion Benchmark\n'
                 '(b,c,e,f use same color scale 0–50 m for direct comparison)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    panel_path = os.path.join(results_dir, f'frame_{idx:06d}_benchmark.png')
    plt.savefig(panel_path, dpi=120, bbox_inches='tight')
    plt.close(fig)

    # ---- SIDE-BY-SIDE FOR GIF (Camera + Fused Depth) ----
    fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Camera Image', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    im = axes[1].imshow(fused_depth, cmap='plasma', vmin=0, vmax=50)
    axes[1].set_title('Fused Depth (LiDAR + MiDaS)', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.suptitle(f'Frame {idx:06d}')
    plt.tight_layout()
    gif_frame_path = os.path.join(gif_frames_dir, f'frame_{idx:06d}.png')
    plt.savefig(gif_frame_path, dpi=100, bbox_inches='tight')
    plt.close(fig2)

print("\n✅ Processing complete. All benchmark frames saved.")


🔄 Processing 10 frames and saving results...

  Frame 000000


/tmp/ipykernel_11251/3145819961.py:108: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_11251/3145819961.py:110: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(panel_path, dpi=120, bbox_inches='tight')


  Frame 000001
  Frame 000002
  Frame 000003
  Frame 000004
  Frame 000005
  Frame 000006
  Frame 000007
  Frame 000008
  Frame 000009

✅ Processing complete. All frames saved.
